# Stomp Pad Studio: board detector training

This notebook trains the small neural network that finds a snowboard in a photo, using the photos people share from the website.

1. **Load** the shared photos and their corrected board outlines from Google Drive.
2. **Measure** how well the current colour-based detector does (the baseline).
3. **Train** a tiny U-Net (about 40k parameters) that outputs a board mask.
4. **Compare** it with the baseline on photos it never saw.
5. **Export** `board-model.json` for the website, which runs it in the browser with no server.

Run it in Google Colab (*Runtime → Change runtime type → GPU* helps, but a CPU works too). It needs **at least ~30 boards** to be useful; more than 100 is better.

In [ ]:
# --- Settings -------------------------------------------------------------------------
DATA_DIR = '/content/drive/MyDrive/Stomp Pad Dataset'   # folder made by the Apps Script
SIZE = 128              # network input (must stay a multiple of 8)
CHANNELS = [8, 16, 32, 32]
EPOCHS = 250
VAL_FRACTION = 0.2
SEED = 0
MODEL_VERSION = 'unet-v1'

In [ ]:
import os, sys, json, glob, base64, math, random
import numpy as np
from PIL import Image, ImageDraw, ImageEnhance
import matplotlib.pyplot as plt
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
random.seed(SEED); np.random.seed(SEED)
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

## 1. Load the dataset

In [ ]:
def poly_mask(outline, w, h):
    m = Image.new('L', (w, h), 0)
    ImageDraw.Draw(m).polygon([(x * w, y * h) for x, y in outline], fill=255)
    return np.array(m) > 127

def load_dataset(root):
    items = []
    for lp in sorted(glob.glob(os.path.join(root, 'labels', '*.json'))):
        lab = json.load(open(lp))
        ip = os.path.join(root, 'images', lab['id'] + '.jpg')
        if not os.path.exists(ip) or len(lab['final']['outline']) < 3: continue
        img = Image.open(ip).convert('RGB')
        s = 256 / max(img.size)                       # keep a small working copy
        img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), Image.BOX)
        lab['mask'] = poly_mask(lab['final']['outline'], img.width, img.height)
        lab['auto_mask'] = poly_mask(lab['auto']['outline'], img.width, img.height) if lab.get('auto') else None
        lab['img'] = img
        items.append(lab)
    return items

data = load_dataset(DATA_DIR)
print(f'{len(data)} boards loaded')
if data:
    print('user-corrected:', sum(d['corrected'] for d in data), '| board lengths (cm):', sorted({d['lengthCm'] for d in data}))

In [ ]:
# A look at the data: photo with the labelled outline in green
n = min(8, len(data))
if n:
    fig, axs = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, d in zip(np.atleast_1d(axs), data[:n]):
        ax.imshow(d['img']); ax.contour(d['mask'], [0.5], colors='lime', linewidths=1.5); ax.axis('off')
        ax.set_title(f"{d['lengthCm']} cm, waist {d['waistMm']/10:.1f} cm", fontsize=9)
    plt.tight_layout(); plt.show()

## 2. Baseline: how good is the current colour-based detector?

In [ ]:
def iou(a, b):
    u = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / u) if u else 1.0

def waist_mm_from_mask(m, length_cm):
    """PCA axis -> length in px; narrowest width over the middle 25-75% -> waist in mm."""
    ys, xs = np.nonzero(m)
    if len(xs) < 50: return float('nan')
    P = np.stack([xs, ys], 1).astype(float); mu = P.mean(0)
    w_, v = np.linalg.eigh(np.cov((P - mu).T)); u, p = v[:, 1], v[:, 0]
    a, b = (P - mu) @ u, (P - mu) @ p
    a0, a1 = np.percentile(a, [0.5, 99.5]); L = a1 - a0
    widths = []
    for t in np.linspace(0.25, 0.75, 21):
        sel = np.abs(a - (a0 + t * L)) < L / 80
        if sel.sum() > 3: widths.append(np.percentile(b[sel], 99) - np.percentile(b[sel], 1))
    return float(np.median(sorted(widths)[:5]) / L * length_cm * 10) if widths else float('nan')

base_iou = [iou(d['auto_mask'], d['mask']) for d in data if d['auto_mask'] is not None]
base_err = [abs(d['auto']['waistMm'] - d['waistMm']) for d in data if d.get('auto') and d['auto'].get('waistMm')]
if base_iou:
    print(f'Colour detector: mean IoU {np.mean(base_iou):.3f}, found the board (IoU>0.8) in {np.mean(np.array(base_iou) > 0.8) * 100:.0f}% of photos')
    print(f'Colour detector: median waist error {np.median(base_err):.1f} mm' if base_err else '')
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].hist(base_iou, bins=20, range=(0, 1), color='#0e1a2b'); ax[0].set_xlabel('IoU with corrected outline'); ax[0].set_ylabel('photos'); ax[0].set_title('Colour detector: outline accuracy')
    if base_err: ax[1].hist(base_err, bins=20, color='#3d8bff'); ax[1].set_xlabel('waist error (mm)'); ax[1].set_title('Colour detector: waist error')
    plt.tight_layout(); plt.show()

## 3. Preprocessing and augmentation
The network sees a 128×128 letterboxed image, exactly like the website: long side scaled to 128 with area averaging, at the top-left, padded with the mean colour.

In [ ]:
def letterbox(img, mask=None, S=SIZE):
    k = S / max(img.size); iw, ih = max(1, round(img.width * k)), max(1, round(img.height * k))
    x = np.tile((MEAN * 255).astype(np.float32), (S, S, 1))
    x[:ih, :iw] = np.asarray(img.resize((iw, ih), Image.BOX), np.float32)
    x = ((x / 255 - MEAN) / STD).transpose(2, 0, 1).astype(np.float32)
    if mask is None: return x
    m = np.zeros((S, S), np.float32)
    m[:ih, :iw] = np.asarray(Image.fromarray((mask * 255).astype(np.uint8)).resize((iw, ih), Image.BOX), np.float32) / 255
    return x, (m > 0.5).astype(np.float32)

def augment(img, mask):
    """Boards are photographed at any angle, in any light, on any floor."""
    m = Image.fromarray((mask * 255).astype(np.uint8))
    ang = random.uniform(0, 360)
    img = img.rotate(ang, Image.BILINEAR, expand=True, fillcolor=tuple(int(v) for v in np.asarray(img).reshape(-1, 3).mean(0)))
    m = m.rotate(ang, Image.NEAREST, expand=True)
    if random.random() < 0.5: img, m = img.transpose(Image.FLIP_LEFT_RIGHT), m.transpose(Image.FLIP_LEFT_RIGHT)
    s = random.uniform(0.75, 1.0)                     # random crop that keeps most of the board
    W, H = img.size; cw, ch = int(W * s), int(H * s); x0, y0 = random.randint(0, W - cw), random.randint(0, H - ch)
    img, m = img.crop((x0, y0, x0 + cw, y0 + ch)), m.crop((x0, y0, x0 + cw, y0 + ch))
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.6, 1.4))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.7, 1.3))
    img = ImageEnhance.Color(img).enhance(random.uniform(0.6, 1.4))
    if random.random() < 0.3:                          # hue shift: a yellow board should not be special
        hsv = np.array(img.convert('HSV')); hsv[..., 0] = (hsv[..., 0].astype(int) + random.randint(0, 255)) % 256
        img = Image.fromarray(hsv, 'HSV').convert('RGB')
    return img, np.asarray(m) > 127

if data:
    fig, axs = plt.subplots(1, 6, figsize=(15, 2.6))
    for ax in axs:
        im, mk = augment(data[0]['img'], data[0]['mask']); x, y = letterbox(im, mk)
        ax.imshow(np.clip(x.transpose(1, 2, 0) * STD + MEAN, 0, 1)); ax.contour(y, [0.5], colors='lime'); ax.axis('off')
    plt.suptitle('Augmented training samples'); plt.show()

## 4. The model
A tiny U-Net. BatchNorm is folded into the convolutions when exporting, so the website only needs convolution, ReLU, max-pool and upsampling.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)

def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
class TinyUNet(nn.Module):
    def __init__(s, ch=CHANNELS):
        super().__init__(); c1, c2, c3, c4 = ch
        s.enc1 = nn.Sequential(block(3, c1), block(c1, c1)); s.enc2 = nn.Sequential(block(c1, c2), block(c2, c2))
        s.enc3 = nn.Sequential(block(c2, c3), block(c3, c3)); s.bott = nn.Sequential(block(c3, c4), block(c4, c4))
        s.dec3 = nn.Sequential(block(c4 + c3, c3), block(c3, c3)); s.dec2 = nn.Sequential(block(c3 + c2, c2), block(c2, c2))
        s.dec1 = nn.Sequential(block(c2 + c1, c1), block(c1, c1)); s.out = nn.Conv2d(c1, 1, 1)
    def forward(s, x):
        up = lambda t: F.interpolate(t, scale_factor=2, mode='nearest')
        e1 = s.enc1(x); e2 = s.enc2(F.max_pool2d(e1, 2)); e3 = s.enc3(F.max_pool2d(e2, 2)); b = s.bott(F.max_pool2d(e3, 2))
        d = s.dec3(torch.cat([up(b), e3], 1)); d = s.dec2(torch.cat([up(d), e2], 1)); d = s.dec1(torch.cat([up(d), e1], 1))
        return s.out(d)[:, 0]

model = TinyUNet().to(dev)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

## 5. Train

In [ ]:
assert len(data) >= 10, f'Only {len(data)} boards so far: collect at least ~30 shared photos before training.'
idx = list(range(len(data))); random.Random(SEED).shuffle(idx)
n_val = max(1, int(len(idx) * VAL_FRACTION)) if len(idx) > 4 else 0
val_idx, train_idx = idx[:n_val], idx[n_val:]
print(f'train {len(train_idx)} / validation {len(val_idx)}')

def batch(ids, aug):
    xs, ys = [], []
    for i in ids:
        im, mk = (augment(data[i]['img'], data[i]['mask']) if aug else (data[i]['img'], data[i]['mask']))
        x, y = letterbox(im, mk); xs.append(x); ys.append(y)
    return torch.tensor(np.stack(xs)).to(dev), torch.tensor(np.stack(ys)).to(dev)

def loss_fn(logit, y):
    p = torch.sigmoid(logit); dice = 1 - (2 * (p * y).sum((1, 2)) + 1) / (p.sum((1, 2)) + y.sum((1, 2)) + 1)
    return F.binary_cross_entropy_with_logits(logit, y) + dice.mean()

opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=EPOCHS)
BS = 16; hist = {'train': [], 'val': []}
for ep in range(EPOCHS):
    model.train(); random.shuffle(train_idx); tl = []
    for k in range(0, len(train_idx), BS):
        x, y = batch(train_idx[k:k + BS], True)
        loss = loss_fn(model(x), y); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
    sched.step(); hist['train'].append(np.mean(tl))
    if val_idx and (ep % 10 == 9 or ep == EPOCHS - 1):
        model.eval()
        with torch.no_grad():
            x, y = batch(val_idx, False); pv = (torch.sigmoid(model(x)) > 0.5).float()
            hist['val'].append((ep, ((pv * y).sum((1, 2)) / ((pv + y) > 0).float().sum((1, 2)).clamp(min=1)).mean().item()))
        print(f'epoch {ep + 1:4d}  loss {hist["train"][-1]:.3f}  val IoU {hist["val"][-1][1]:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist['train']); ax[0].set_title('training loss'); ax[0].set_xlabel('epoch')
if hist['val']: ax[1].plot(*zip(*hist['val']), 'o-'); ax[1].set_title('validation IoU'); ax[1].set_xlabel('epoch'); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 6. Compare with the colour detector (validation photos only)

In [ ]:
def predict(img):
    model.eval(); x = letterbox(img)
    with torch.no_grad(): p = torch.sigmoid(model(torch.tensor(x)[None].to(dev)))[0].cpu().numpy()
    k = SIZE / max(img.size); iw, ih = max(1, round(img.width * k)), max(1, round(img.height * k))
    small = (p[:ih, :iw] > 0.5).astype(np.uint8) * 255
    return np.asarray(Image.fromarray(small).resize(img.size, Image.NEAREST)) > 127

rows = []
for i in val_idx:
    d = data[i]; pm = predict(d['img'])
    rows.append(dict(model_iou=iou(pm, d['mask']), base_iou=iou(d['auto_mask'], d['mask']) if d['auto_mask'] is not None else np.nan,
                     model_waist_err=abs(waist_mm_from_mask(pm, d['lengthCm']) - d['waistMm']),
                     base_waist_err=abs(d['auto']['waistMm'] - d['waistMm']) if d.get('auto') and d['auto'].get('waistMm') else np.nan))
if rows:
    mi, bi = np.array([r['model_iou'] for r in rows]), np.array([r['base_iou'] for r in rows])
    me, be = np.array([r['model_waist_err'] for r in rows]), np.array([r['base_waist_err'] for r in rows])
    print(f'Mean IoU      : model {np.nanmean(mi):.3f}   colour detector {np.nanmean(bi):.3f}')
    print(f'Found (IoU>.8): model {np.mean(mi > 0.8) * 100:.0f}%   colour detector {np.nanmean(bi > 0.8) * 100:.0f}%')
    print(f'Median waist error: model {np.nanmedian(me):.1f} mm   colour detector {np.nanmedian(be):.1f} mm')
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].scatter(bi, mi, c='#0e1a2b'); ax[0].plot([0, 1], [0, 1], '--', c='#aaa'); ax[0].set_xlabel('colour detector IoU'); ax[0].set_ylabel('model IoU'); ax[0].set_title('Per photo: above the line = model better')
    ax[1].bar(['colour detector', 'trained model'], [np.nanmedian(be), np.nanmedian(me)], color=['#7a8699', '#0e1a2b']); ax[1].set_ylabel('median waist error (mm)')
    plt.tight_layout(); plt.show()
    n = min(6, len(val_idx)); fig, axs = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax, i in zip(np.atleast_1d(axs), val_idx[:n]):
        d = data[i]; ax.imshow(d['img']); ax.contour(d['mask'], [0.5], colors='lime'); ax.contour(predict(d['img']), [0.5], colors='red'); ax.axis('off')
    plt.suptitle('green = corrected outline, red = model'); plt.show()

## 7. Export for the website
Folds BatchNorm into the convolutions and writes `board-model.json`. Then a NumPy re-implementation of the website's maths checks the export against PyTorch.

In [ ]:
def fold(seq):
    conv, bn = seq[0], seq[1]
    g = (bn.weight / torch.sqrt(bn.running_var + bn.eps)).detach()
    w = (conv.weight * g[:, None, None, None]).detach().cpu().numpy().astype(np.float32)
    b = (bn.bias - bn.running_mean * g).detach().cpu().numpy().astype(np.float32)
    return w, b
names = {'enc1': ('enc1a', 'enc1b'), 'enc2': ('enc2a', 'enc2b'), 'enc3': ('enc3a', 'enc3b'), 'bott': ('botta', 'bottb'),
         'dec3': ('dec3a', 'dec3b'), 'dec2': ('dec2a', 'dec2b'), 'dec1': ('dec1a', 'dec1b')}
model.eval(); W = {}
for mod, (na, nb) in names.items():
    seq = getattr(model, mod); W[na] = fold(seq[0]); W[nb] = fold(seq[1])
W['out'] = (model.out.weight.detach().cpu().numpy().astype(np.float32), model.out.bias.detach().cpu().numpy().astype(np.float32))
enc = lambda a: base64.b64encode(np.ascontiguousarray(a, np.float32).tobytes()).decode()
export = {'version': MODEL_VERSION, 'size': SIZE, 'mean': MEAN.tolist(), 'std': STD.tolist(), 'ch': CHANNELS,
          'trained_on': len(train_idx), 'layers': {k: {'w': enc(w), 'b': enc(b), 'shape': list(w.shape)} for k, (w, b) in W.items()}}

# NumPy forward = exactly what core.js does in the browser
def np_conv(x, w, b, relu):
    O, I, K, _ = w.shape; C, H, Wd = x.shape; r = K // 2; xp = np.pad(x, ((0, 0), (r, r), (r, r)))
    out = np.tile(b[:, None, None], (1, H, Wd)).astype(np.float32)
    for ky in range(K):
        for kx in range(K): out += np.einsum('oi,ihw->ohw', w[:, :, ky, kx], xp[:, ky:ky + H, kx:kx + Wd])
    return np.maximum(out, 0) if relu else out
def np_forward(x):
    c = lambda t, n: np_conv(t, *W[n], True)
    pool = lambda t: t.reshape(t.shape[0], t.shape[1] // 2, 2, t.shape[2] // 2, 2).max(axis=(2, 4))
    upcat = lambda t, s: np.concatenate([t.repeat(2, 1).repeat(2, 2), s], 0)
    e1 = c(c(x, 'enc1a'), 'enc1b'); e2 = c(c(pool(e1), 'enc2a'), 'enc2b'); e3 = c(c(pool(e2), 'enc3a'), 'enc3b')
    bt = c(c(pool(e3), 'botta'), 'bottb'); d = c(c(upcat(bt, e3), 'dec3a'), 'dec3b')
    d = c(c(upcat(d, e2), 'dec2a'), 'dec2b'); d = c(c(upcat(d, e1), 'dec1a'), 'dec1b')
    return 1 / (1 + np.exp(-np_conv(d, *W['out'], False)[0]))

x = letterbox(data[(val_idx or train_idx)[0]]['img'])
with torch.no_grad(): pt = torch.sigmoid(model(torch.tensor(x)[None].to(dev)))[0].cpu().numpy()
diff = np.abs(np_forward(x) - pt).max(); print(f'export check: max difference {diff:.2e}'); assert diff < 1e-3

out_path = os.path.join(DATA_DIR, 'board-model.json')
json.dump(export, open(out_path, 'w')); print('saved', out_path, f'({os.path.getsize(out_path) / 1024:.0f} KB)')
if IN_COLAB:
    from google.colab import files; files.download(out_path)

## 8. Put it on the website
Copy `board-model.json` to `stomp-pad/model/board-model.json` in the GitHub repo and commit, or send it to Claude. The site loads it automatically and shows "trained model" when it finds a board with it. If the model can't find a board, the site falls back to the colour detector.

Retrain whenever more boards come in. Each new photo shared from the site lands in the same Drive folder.